In [41]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,accuracy_score
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [42]:
df = pd.read_csv("ecommerce_user_behavior_8000.csv")

In [43]:
df.head()

,user_id,age,gender,device_type,time_on_site,pages_viewed,previous_purchases,cart_items,discount_seen,ad_clicked,returning_user,avg_session_time,bounce_rate,purchase
0,1.0,56.0,Female,Desktop,12.90,8.0,13.0,1.0,1.0,NaN,0.0,6.97,28.18,1.0
1,2.0,46.0,Male,Mobile,15.63,9.0,4.0,6.0,1.0,1.0,1.0,19.17,86.73,1.0
2,3.0,32.0,Female,NaN,11.64,12.0,11.0,0.0,0.0,0.0,1.0,8.87,83.09,1.0
3,4.0,25.0,Female,Mobile,22.71,5.0,10.0,1.0,0.0,0.0,1.0,NaN,79.03,1.0
4,5.0,38.0,Female,Mobile,26.35,9.0,12.0,4.0,1.0,0.0,0.0,18.15,55.35,1.0


In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             7840 non-null   float64
 1   age                 7840 non-null   float64
 2   gender              7840 non-null   object 
 3   device_type         7840 non-null   object 
 4   time_on_site        7840 non-null   float64
 5   pages_viewed        7840 non-null   float64
 6   previous_purchases  7840 non-null   float64
 7   cart_items          7840 non-null   float64
 8   discount_seen       7840 non-null   float64
 9   ad_clicked          7840 non-null   float64
 10  returning_user      7840 non-null   float64
 11  avg_session_time    7840 non-null   float64
 12  bounce_rate         7840 non-null   float64
 13  purchase            7840 non-null   float64
dtypes: float64(12), object(2)
memory usage: 875.1+ KB


In [45]:
df.head()

,user_id,age,gender,device_type,time_on_site,pages_viewed,previous_purchases,cart_items,discount_seen,ad_clicked,returning_user,avg_session_time,bounce_rate,purchase
0,1.0,56.0,Female,Desktop,12.90,8.0,13.0,1.0,1.0,NaN,0.0,6.97,28.18,1.0
1,2.0,46.0,Male,Mobile,15.63,9.0,4.0,6.0,1.0,1.0,1.0,19.17,86.73,1.0
2,3.0,32.0,Female,NaN,11.64,12.0,11.0,0.0,0.0,0.0,1.0,8.87,83.09,1.0
3,4.0,25.0,Female,Mobile,22.71,5.0,10.0,1.0,0.0,0.0,1.0,NaN,79.03,1.0
4,5.0,38.0,Female,Mobile,26.35,9.0,12.0,4.0,1.0,0.0,0.0,18.15,55.35,1.0


In [46]:
df['purchase'].unique()

array([ 1., nan,  0.])

In [47]:
df.isna().sum()

user_id               160
age                   160
gender                160
device_type           160
time_on_site          160
pages_viewed          160
previous_purchases    160
cart_items            160
discount_seen         160
ad_clicked            160
returning_user        160
avg_session_time      160
bounce_rate           160
purchase              160
dtype: int64

In [48]:
df.shape

(8000, 14)

In [49]:
df = df.dropna(subset=['purchase'])

In [50]:
df.isna().sum()

user_id               154
age                   156
gender                156
device_type           156
time_on_site          159
pages_viewed          156
previous_purchases    155
cart_items            159
discount_seen         157
ad_clicked            158
returning_user        158
avg_session_time      158
bounce_rate           156
purchase                0
dtype: int64

In [51]:
df1 = df.drop(columns=['user_id'],axis=1)

In [52]:
df1.isna().sum()

age                   156
gender                156
device_type           156
time_on_site          159
pages_viewed          156
previous_purchases    155
cart_items            159
discount_seen         157
ad_clicked            158
returning_user        158
avg_session_time      158
bounce_rate           156
purchase                0
dtype: int64

In [53]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7840 entries, 0 to 7999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   age                 7684 non-null   float64
 1   gender              7684 non-null   object 
 2   device_type         7684 non-null   object 
 3   time_on_site        7681 non-null   float64
 4   pages_viewed        7684 non-null   float64
 5   previous_purchases  7685 non-null   float64
 6   cart_items          7681 non-null   float64
 7   discount_seen       7683 non-null   float64
 8   ad_clicked          7682 non-null   float64
 9   returning_user      7682 non-null   float64
 10  avg_session_time    7682 non-null   float64
 11  bounce_rate         7684 non-null   float64
 12  purchase            7840 non-null   float64
dtypes: float64(11), object(2)
memory usage: 857.5+ KB


In [54]:
num_feature = ['age','time_on_site','pages_viewed','previous_purchases','cart_itmes','discount_seen','ad_clicked','avg_session_time','bounce_rate']

In [55]:
cat_feature = ['gender','device_type']

In [56]:
num_pipeline = Pipeline([
    ('impute',SimpleImputer(strategy='mean'))
])

In [57]:
num_pipeline

Pipeline(steps=[('impute', SimpleImputer())])

In [58]:
cat_pipeline = Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('ohe',OneHotEncoder(handle_unknown='infrequent_if_exist'))
])

In [59]:
cat_pipeline

Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='infrequent_if_exist'))])

In [60]:
preprocessor = ColumnTransformer(transformers=[
    (num_pipeline,num_feature),
    (cat_pipeline,cat_feature)
])

In [61]:
preprocessor

ColumnTransformer(transformers=[(Pipeline(steps=[('impute', SimpleImputer())]),
                                 ['age', 'time_on_site', 'pages_viewed',
                                  'previous_purchases', 'cart_itmes',
                                  'discount_seen', 'ad_clicked',
                                  'avg_session_time', 'bounce_rate']),
                                (Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'))]),
                                 ['gender', 'device_type'])])

In [62]:
x = df1.drop(columns='purchase',axis=1)
y = df1['purchase']

In [63]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [64]:
xg = XGBClassifier()

In [65]:
final_workflow = Pipeline([
    ('preprocessor',preprocessor),
    ('model',xg)
])

In [66]:
final_workflow

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[(Pipeline(steps=[('impute',
                                                                   SimpleImputer())]),
                                                  ['age', 'time_on_site',
                                                   'pages_viewed',
                                                   'previous_purchases',
                                                   'cart_itmes',
                                                   'discount_seen',
                                                   'ad_clicked',
                                                   'avg_session_time',
                                                   'bounce_rate']),
                                                 (Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, random_state=None, ...))])

In [67]:
final_workflow.fit(x_train,y_train)

ValueError: not enough values to unpack (expected 3, got 2)